# Loan approval prediction: reproduction notebook

This notebook re-runs the whole evaluation using the repository's code and checks that every number matches the results saved in `reports/`. It never overwrites anything in `reports/`.

**Time:** about 4 minutes (the SVM is the slow part). **Needs:** `loan_data.csv`, see [DATA.md](https://github.com/zohabaamir-ai/loan-approval-prediction-system/blob/main/DATA.md).

**In Google Colab:** choose *Runtime > Run all*. The first cell installs the pinned library versions. If it tells you to restart the session, do that (*Runtime > Restart session*) and run all cells again. When asked, upload `loan_data.csv`. A plain CPU runtime is enough; a GPU is not needed.

**Locally:** open it with the project's `.venv` as the kernel (see the README quick start).

> The dataset is synthetic and the label is a historical decision, not repayment. This is a demo of an evaluation method, not a lending tool. See the README's *Fairness and limitations* section.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/zohabaamir-ai/loan-approval-prediction-system.git"
REPO_DIR = "loan-approval-prediction-system"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Fetch the repository, install the pinned versions, and make the package importable.
    if not os.path.exists(REPO_DIR) and not os.path.exists("pyproject.toml"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    if os.path.exists(REPO_DIR):
        os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    sys.path.insert(0, os.path.abspath("src"))
    for pip_name, module_name, wanted in (
        ("pandas", "pandas", "2.2.3"),
        ("scikit-learn", "sklearn", "1.5.2"),
    ):
        loaded = sys.modules.get(module_name)
        if loaded is not None and loaded.__version__ != wanted:
            raise RuntimeError(
                f"{pip_name} {loaded.__version__} is already loaded but {wanted} is needed. "
                "Choose Runtime > Restart session, then run all cells again."
            )
print("Running in Colab:", IN_COLAB)

In [ ]:
import json
import time

import matplotlib.pyplot as plt
import pandas as pd
import sklearn
from IPython.display import display

from loan_approval.candidates import (
    BASELINE_NAME,
    CHALLENGER_NAMES,
    FALLBACK_NAME,
    beats_baseline,
    build_candidate,
)
from loan_approval.data import DEFAULT_DATA_PATH, REPO_ROOT, clean_data, load_data, split_data
from loan_approval.evaluate import SCORING, cross_validate_model, format_table
from loan_approval.fairness import age_band, group_report
from loan_approval.final import bootstrap_ci, score_at_threshold
from loan_approval.pipeline import MODEL_NAMES, REFERENCE_NAMES, build_pipeline, build_reference
from loan_approval.threshold import best_threshold, metrics_at, oof_probabilities, threshold_table

print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__)


def saved(name):
    """A result file committed in the repository."""
    return json.loads((REPO_ROOT / "reports" / name).read_text(encoding="utf-8"))


def compare(label, fresh, saved_results, tol=0.002):
    """Largest difference between the metric means computed here and the saved ones."""
    worst = max(
        abs(res[m]["mean"] - saved_results[name][m]["mean"])
        for name, res in fresh.items()
        for m in SCORING
    )
    verdict = "matches" if worst <= tol else "DIFFERS from"
    print(f"\n{label}: {verdict} the saved result (largest difference in a mean: {worst:.4f})")

## 1. Data

The raw data is not in the repository. In Colab you will be asked to upload `loan_data.csv`; locally it must already be at `data/loan_data.csv`.

In [ ]:
if not DEFAULT_DATA_PATH.exists() and IN_COLAB:
    from google.colab import files

    uploaded = files.upload()  # choose loan_data.csv
    DEFAULT_DATA_PATH.parent.mkdir(exist_ok=True)
    DEFAULT_DATA_PATH.write_bytes(next(iter(uploaded.values())))

df = clean_data(load_data())
X_train, X_test, y_train, y_test = split_data(df)  # split FIRST, before anything is fitted
assert (len(df), len(X_train), len(X_test)) == (44993, 35994, 8999), (
    "Unexpected row counts: is this the right file? See DATA.md."
)
print(f"rows after cleaning: {len(df):,} | train: {len(X_train):,} | test: {len(X_test):,}")
print(f"approved share: train {y_train.mean():.3f}, test {y_test.mean():.3f}")
print(f"always predicting 'rejected' would score {1 - y_train.mean():.3f} accuracy")

## 2. Honest baseline

The four original models, each inside a scikit-learn Pipeline (imputing, scaling and one-hot encoding are fitted on training folds only), plus two reference points. 5-fold stratified cross-validation on the **training split only**. The test split is not touched in this section. The SVM takes about two minutes.

In [ ]:
results = {}
for name in REFERENCE_NAMES:
    results[name] = cross_validate_model(build_reference(name), X_train, y_train)
for name in MODEL_NAMES:
    start = time.time()
    results[name] = cross_validate_model(build_pipeline(name), X_train, y_train)
    print(f"done {name:<22} {time.time() - start:6.1f}s")

print()
print(format_table(results))
compare("Baseline", results, saved("baseline_cv.json")["results"])

## 3. One stronger model, and the adoption rule

HistGradientBoosting without the gender feature, default settings, no tuning. The rule was fixed before running it: adopt it only if its mean cross-validated ROC-AUC beats the Random Forest baseline's mean **plus** that baseline's fold-to-fold standard deviation. Otherwise keep the simple Random Forest without gender.

In [ ]:
cand = {}
for name in (FALLBACK_NAME, *CHALLENGER_NAMES):
    cand[name] = cross_validate_model(build_candidate(name), X_train, y_train)

print(format_table({BASELINE_NAME + " (baseline)": results[BASELINE_NAME], **cand}))

final_name = FALLBACK_NAME
for name in CHALLENGER_NAMES:
    d = beats_baseline(results[BASELINE_NAME], cand[name])
    print(f"\n{name}: needs > {d['required_above']:.4f}, got {d['candidate_mean']:.4f}")
    if d["wins"]:
        final_name = name
print("\nFinal model:", final_name)
compare("Candidates", cand, saved("candidates_cv.json")["results"])

## 4. Decision threshold

Chosen on **out-of-fold predictions on the training data**: every training row is scored by a model that never saw it. The assumption is that a false approval costs three times a false rejection. Remember that the label is a historical decision, so "false approval" is only a stand-in for a bad loan.

In [ ]:
proba_oof = oof_probabilities(build_candidate(final_name), X_train, y_train)
chosen = best_threshold(y_train, proba_oof, fp_cost=3.0, fn_cost=1.0)
default = metrics_at(y_train, proba_oof, 0.5)

for label, m in (("default 0.50", default), ("chosen, 3:1 cost", chosen)):
    print(
        f"{label:<18} thr={m['threshold']:.2f}  precision={m['precision']:.3f}  "
        f"recall={m['recall']:.3f}  false approvals={m['fp']}  false rejections={m['fn']}"
    )
saved_threshold = saved("threshold.json")["chosen"]["threshold"]
print(f"\nsaved threshold: {saved_threshold:.2f} | recomputed here: {chosen['threshold']:.2f}")

table = threshold_table(y_train, proba_oof)
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(table["threshold"], table["precision"], label="precision")
ax.plot(table["threshold"], table["recall"], label="recall")
ax.plot(table["threshold"], table["f1"], label="F1", linestyle=":")
ax.axvline(chosen["threshold"], color="grey", linestyle="--", label="chosen threshold")
ax.set_xlabel("decision threshold (approve if probability >= threshold)")
ax.set_ylabel("score (out-of-fold, training data)")
ax.set_title("Precision / recall trade-off")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 5. Held-out test split: reproducing the one-time score

This is a **reproduction check, not a new evaluation**. The model and the threshold are frozen: the threshold is read from the saved `reports/threshold.json`, which was written before the test split was ever scored. Nothing is chosen or tuned using these results. The cell fits the final model on the training split, scores the test split, and compares with the saved `reports/final_test.json`.

In [ ]:
threshold = saved("threshold.json")["chosen"]["threshold"]  # frozen before the test split was used
model = build_candidate(final_name).fit(X_train, y_train)
proba_test = model.predict_proba(X_test)[:, 1]

test = score_at_threshold(y_test, proba_test, threshold)
ci = bootstrap_ci(y_test, proba_test, threshold)  # 2,000 resamples, takes a few seconds

metrics = ["roc_auc", "precision", "recall", "f1", "accuracy"]
summary = pd.DataFrame(
    {
        "value": [round(test[m], 3) for m in metrics],
        "95% CI low": [round(ci[m]["low"], 3) for m in metrics],
        "95% CI high": [round(ci[m]["high"], 3) for m in metrics],
    },
    index=metrics,
)
display(summary)

saved_test = saved("final_test.json")["test"]
worst = max(abs(test[m] - saved_test[m]) for m in metrics)
verdict = "matches" if worst <= 0.002 else "DIFFERS from"
print(f"\nTest result {verdict} the saved result (largest difference: {worst:.4f})")

counts = [[test["tn"], test["fp"]], [test["fn"], test["tp"]]]
fig, ax = plt.subplots(figsize=(4.8, 4.2))
ax.imshow(counts, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{counts[i][j]:,}", ha="center", va="center", fontsize=14)
ax.set_xticks([0, 1], ["rejected", "approved"])
ax.set_yticks([0, 1], ["rejected", "approved"])
ax.set_xlabel("model decision")
ax.set_ylabel("historical decision")
ax.set_title(f"Held-out test, threshold {threshold:.2f}")
plt.show()

## 6. Fairness audit

Gender is not a model feature, but we still check the outcome by gender. Age is a model feature, so we check it by age band. This looks at a few standard gaps only; it is a check, not a certificate of fairness. The 40-and-over group is small, so its numbers are noisy.

In [ ]:
gender_report = group_report(y_test, proba_test, X_test["person_gender"], threshold)
age_report = group_report(y_test, proba_test, age_band(X_test["person_age"]), threshold)

print("By gender:")
display(pd.DataFrame(gender_report).T.round(3))
print("By age band:")
display(pd.DataFrame(age_report).T.round(3))

## What to take from this

- Compare the tables above with the README: they should agree to about three decimals.
- The ROC-AUC does not depend on the threshold; precision, recall and F1 at 0.78 trade recall for fewer false approvals on purpose.
- The data is synthetic and its patterns are odd (see the README), so these numbers say how learnable the data is, not how a real lending model would perform.